##### 💠 **Build the Two-Site Hamiltonian. A Spin-$\frac{1}{2}$ chain can be modeled as,$$\boxed{H_{n}=-B\sum_{l=1}^{n}\sigma_{z}^{[l]}-J\sum_{l=1}^{n-1}\vec{\sigma}^{[l]}\cdot\vec{\sigma}^{[l+1]}}$$**
Where the
*   **$H_n$**: Represents the Hamiltonian for a chain of $n$ spins.
*   **$B$**: Denotes the strength of the external magnetic field acting in the $z$ direction.
*   **$J$**: Represents the coupling constant for the interaction between adjacent spins.
*   **$\sigma_z^l$**: Is the Pauli-$Z$ matrix or operator for the $l$-th spin.
*   **$\vec{\sigma}_l \cdot \vec{\sigma}_{l+1}$**: Describes the dot product interaction between the vector spin operators of neighboring spins $l$ and $l+1$.

In [3]:
import sys
import os
import time
import numpy as np
sys.path.insert(0, os.path.abspath("../.."))
from src.mps_mpo import (
    init_mps, 
    convert_vidal_to_canonical, 
    build_xxz_mpo, 
    initialize_environments, 
    run_dmrg
)

In [4]:
#---Physical parameter------
n_sites = 14  #L
d       = 2
chi_max = 10
dt      = 0.005
n_steps = 2000
n_sweeps= 6
Jx, Jy, Jz, hz = 4, 4, 4, 2
conv_tol = 1e-12

#----Build MPO-------------
print("\n ▣ Building MPO....",end=" ", flush=True)
mpo_raw =  build_xxz_mpo( L= n_sites, Jx=Jx, Jy=Jy, Jz=Jz, hx=0, hz=hz, S=0.5)
# Prepend None so that site i maps to index i (1-based)
mpo = [None] + mpo_raw
print("done")
print(f"   mpo[1] (left  boundary) : {mpo[1].shape}")
print(f"   mpo[2] (bulk)           : {mpo[2].shape}")
print(f"   mpo[L] (right boundary) : {mpo[n_sites].shape}")

# --- Initial state |↓↓↑↑↑↑↑↑↑↑⟩ ---
print("\n ▣ Building initial MPS ...", end=" ", flush=True)
# config           = [1, 1, 0, 0, 0, 0, 0, 0, 0, 0]
# config           = [("down", 2),("up", n_sites-2)]
config              = [("down",1),("up",1)] * (n_sites//2)
A_list, lam_list = init_mps(config)
left_mps_raw, right_mps_raw = convert_vidal_to_canonical(A_list, lam_list)
# 1-indexed lists — index 0 is a None placeholder
left_mps  = [None] + left_mps_raw
right_mps = [None] + right_mps_raw
print("done")
print(f"   right_mps[1].shape : {right_mps[1].shape}   (χ_L, d, χ_R)")

#------------------Initialise environments---------------------------------------
print("\n ▣ Pre-computing right environments ...", end=" ", flush=True)
left_envs, right_envs = initialize_environments(right_mps, mpo, n_sites)
print("done")
print(f"   left_envs[0]         : {left_envs[0].shape}   (trivial left  boundary)")
print(f"   right_envs[L+1]      : {right_envs[n_sites + 1].shape}   (trivial right boundary)")
print(f"   right_envs[L]        : {right_envs[n_sites].shape}   (after absorbing site L)")
print(f"   right_envs[1]        : {right_envs[1].shape}   (full right environment)")



 ▣ Building MPO.... done
   mpo[1] (left  boundary) : (1, 5, 2, 2)
   mpo[2] (bulk)           : (5, 5, 2, 2)
   mpo[L] (right boundary) : (5, 1, 2, 2)

 ▣ Building initial MPS ... done
   right_mps[1].shape : (1, 2, 1)   (χ_L, d, χ_R)

 ▣ Pre-computing right environments ... done
   left_envs[0]         : (1, 1, 1)   (trivial left  boundary)
   right_envs[L+1]      : (1, 1, 1)   (trivial right boundary)
   right_envs[L]        : (1, 5, 1)   (after absorbing site L)
   right_envs[1]        : (1, 1, 1)   (full right environment)


In [5]:
# ----Run DMRG----------
print(f"\n ▣ Running DMRG  ({n_sweeps} sweeps max) ...")
print("-" * 56)
t0 = time.perf_counter()
E_ground = run_dmrg(
    mps        = left_mps,
    mpo        = mpo,
    left_envs  = left_envs,
    right_envs = right_envs,
    n_sweeps   = n_sweeps,
    chi_max    = chi_max,
    conv_tol   = conv_tol,
    verbose    = True,
)
elapsed = time.perf_counter() - t0
print("-" * 56)


 ▣ Running DMRG  (6 sweeps max) ...
--------------------------------------------------------
Sweep   0 │ E = -24.884597857400 │ ΔE = inf
Sweep   1 │ E = -25.121825414915 │ ΔE = 2.372e-01
Sweep   2 │ E = -25.121907336346 │ ΔE = 8.192e-05
Sweep   3 │ E = -25.121907338934 │ ΔE = 2.588e-09
Sweep   4 │ E = -25.121907338935 │ ΔE = 1.339e-12
Sweep   5 │ E = -25.121907338935 │ ΔE = 1.776e-13
Converged after 6 sweep(s)  (ΔE < 1.0e-12)
--------------------------------------------------------


In [6]:
bond_dims = [left_mps[i].shape[2] for i in range(1, n_sites)]
sep = "=" * 56
print()
print(sep)
print("  Results")
print(sep)
print(f"  Ground-state energy   E   = {E_ground:+.10f}")
print(f"  Energy per site       E/L = {E_ground / n_sites:+.10f}")
print(f"  Wall time                 = {elapsed:.2f} s")
print(f"  Final bond dimensions     = {bond_dims}")
print(f"  Max bond dim reached      = {max(bond_dims)}  (limit: {chi_max})")
print(sep)


  Results
  Ground-state energy   E   = -25.1219073389
  Energy per site       E/L = -1.7944219528
  Wall time                 = 0.44 s
  Final bond dimensions     = [2, 4, 8, 10, 10, 10, 10, 10, 10, 10, 8, 4, 2]
  Max bond dim reached      = 10  (limit: 10)
